# Análisis de Series Temporales con Python

**Mayo 2026 · Bloque V**

## Objetivos
- Preparar datos temporales con índice de fecha
- Crear variables rezagadas y rolling features
- Comparar un baseline con un modelo supervisado de forecasting

## Preparación
Ejecuta la primera celda para cargar librerías. Si falta alguna librería, instálala desde el entorno con `pip install -r requirements.txt`.

## Carga y visualización

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
pd.set_option("display.max_columns", 50)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv(DATA_DIR / "serie_demanda.csv", parse_dates=["fecha"]).set_index("fecha")
display(df.head())
df["demanda"].plot(figsize=(10,4), title="Serie de demanda")
plt.show()

## Baseline ingenuo

In [ ]:
df_eval = df.copy()
df_eval["naive_pred"] = df_eval["demanda"].shift(1)
eval_naive = df_eval.dropna()
print("MAE baseline:", mean_absolute_error(eval_naive["demanda"], eval_naive["naive_pred"]).round(2))

## Features temporales

In [ ]:
feat = df.copy()
for lag in [1, 2, 7, 14, 30]:
    feat[f"lag_{lag}"] = feat["demanda"].shift(lag)
feat["rolling_7"] = feat["demanda"].shift(1).rolling(7).mean()
feat["dia_semana"] = feat.index.dayofweek
feat["mes"] = feat.index.month
feat = feat.dropna()
display(feat.head())

## Random Forest forecasting

In [ ]:
train = feat.iloc[:-30]
test = feat.iloc[-30:]
X_train, y_train = train.drop(columns=["demanda"]), train["demanda"]
X_test, y_test = test.drop(columns=["demanda"]), test["demanda"]

model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE RF:", mean_absolute_error(y_test, pred).round(2))
print("RMSE RF:", mean_squared_error(y_test, pred, squared=False).round(2))

pd.DataFrame({"real": y_test, "predicho": pred}, index=y_test.index).plot(figsize=(10,4), title="Forecasting 30 días")
plt.show()

## Actividad entregable
1. Modifica el dataset o hiperparámetros.
2. Añade una breve interpretación de resultados.
3. Guarda el notebook ejecutado y exporta una versión HTML/PDF si se solicita.